In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_NOTEBOOK_START = Path.cwd().resolve()
for candidate in (_NOTEBOOK_START, *_NOTEBOOK_START.parents):
    if (candidate / "requirements.txt").is_file() and (candidate / "grid_transform").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not locate repo root from {_NOTEBOOK_START}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from grid_transform.notebook_bootstrap import bootstrap_notebook

globals().update(bootstrap_notebook(REPO_ROOT))
print(f"Notebook bootstrap ready: {PROJECT_DIR}")


In [ ]:
# =====================================================================
#  Load Speaker A (nnUNet, frame 143020) and Speaker B (VTNL, 1640)
# =====================================================================

# --- Speaker A: nnUNet contours (.npy) ---
BASE = str(PROJECT_DIR / "vocal-tract-seg")
DATA_DIR_A     = f"{BASE}/data_80/2008-003^01-1791/test"
CONTOURS_DIR_A = f"{BASE}/results/nnunet_080/inference_contours/2008-003^01-1791/test"
FRAME_A = 143020
image_A, contours_A = load_frame_npy(FRAME_A, DATA_DIR_A, CONTOURS_DIR_A)

# --- Speaker B: VTNL (ImageJ ROIs from .zip) ---
VTNL_DIR = str(PROJECT_DIR / "VTNL")
IMAGE_B = "1640_s10_0654"
image_B, contours_B = load_frame_vtnl(IMAGE_B, VTNL_DIR)

In [ ]:
# =====================================================================
#  Build grids for Speaker A and Speaker B
# =====================================================================
grid_A = build_grid(image_A, contours_A, n_vert=9, n_points=250, frame_number=FRAME_A)
grid_B = build_grid(image_B, contours_B, n_vert=9, n_points=250, frame_number=0)

print_grid_summary(grid_A)
print_grid_summary(grid_B)

In [ ]:
# =====================================================================
#  Visualize both grids side by side
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(32, 16))

visualize_grid(grid_A, ax=ax1, show_labels=True)
ax1.set_title(f"Speaker A (nnUNet) – Frame {FRAME_A}\n6 horiz × 9 vert grid", fontsize=16, fontweight='bold')

visualize_grid(grid_B, ax=ax2, show_labels=True)
ax2.set_title(f"Speaker B (VTNL) – {IMAGE_B}\n6 horiz × 9 vert grid", fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  H1 composite curve visualization  (L1 -> I5 -> C1)
#  — blended overlap path: midline clamped to max deviation from chord
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(32, 16))

for ax, g, spk in zip(axes, [grid_A, grid_B], ['Speaker A', 'Speaker B']):
    # Background image
    ax.imshow(g.image, cmap='gray', origin='upper')

    # Draw all horizontal lines (thin gray) for context
    for k, h in enumerate(g.horiz_lines):
        if k > 0:
            ax.plot(h[:, 0], h[:, 1], '-', color='gray', alpha=0.3, lw=0.8)

    # ---------- H1 anatomy ----------
    h1 = g.horiz_lines[0]
    idx = g.palate_end_idx  # I5 position on H1

    # Segment 1  (L1 -> I5): palate contour — lime
    ax.plot(h1[:idx+1, 0], h1[:idx+1, 1], '-', color='lime',
            lw=3, label=f'L1->I5  palate seg. ({idx+1} pts)')

    # Segment 2  (I5 -> C1): blended overlap path — magenta
    clamp_pct = g.h1_max_clamp * 100
    lbl2 = f'I5->C1  overlap (clamped {clamp_pct:.0f}%)'
    ax.plot(h1[idx:, 0], h1[idx:, 1], '-', color='magenta', lw=3, label=lbl2)

    # Show direct chord I5->C1 for reference (dashed white)
    I5 = g.I_points['I5']
    ax.plot([I5[0], h1[-1, 0]], [I5[1], h1[-1, 1]], '--',
            color='white', lw=1, alpha=0.5, label='I5->C1 direct chord')

    # ---------- VT curve (reference, dashed cyan) ----------
    ax.plot(g.vt_curve[:, 0], g.vt_curve[:, 1], '--', color='cyan',
            lw=1.2, alpha=0.6, label='VT curve (midline ref)')

    # ---------- Landmarks ----------
    if g.I_points:
        for name in ['I1', 'I5']:
            pt = g.I_points[name]
            ax.plot(pt[0], pt[1], 'o', color='yellow', ms=10, zorder=5)
            ax.annotate(name, xy=(pt[0], pt[1]),
                        xytext=(5, -12), textcoords='offset points',
                        fontsize=11, color='yellow', fontweight='bold')
    if g.P1_point is not None:
        ax.plot(g.P1_point[0], g.P1_point[1], 's', color='orange', ms=10, zorder=5)
        ax.annotate('P1', xy=(g.P1_point[0], g.P1_point[1]),
                    xytext=(5, -12), textcoords='offset points',
                    fontsize=11, color='orange', fontweight='bold')
    # C1 = right endpoint of H1
    c1_pt = h1[-1]
    ax.plot(c1_pt[0], c1_pt[1], 'D', color='white', ms=9, zorder=5)
    ax.annotate('C1', xy=(c1_pt[0], c1_pt[1]),
                xytext=(5, -12), textcoords='offset points',
                fontsize=11, color='white', fontweight='bold')

    ax.set_title(
        f"{spk}  —  H1 overlap path\n"
        f"dev_ratio = {g.h1_dev_ratio:.3f}  |  clamped = {clamp_pct:.0f}%",
        fontsize=14, fontweight='bold')
    ax.legend(loc='lower left', fontsize=11, framealpha=0.8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  Method 1: Spine-based alignment  (C1..C6)
# =====================================================================

def extract_true_landmarks(grid):
    """Collect the landmark set used by the alignment experiments."""
    C = np.array([grid.cervical_centers[f'c{i}'] for i in range(1, 7)], dtype=float)
    return {
        'C': C,
        'L1': np.asarray(grid.left_pts[0], dtype=float),
        'L6': np.asarray(grid.left_pts[-1], dtype=float),
        'M1': np.asarray(grid.M1_point, dtype=float) if grid.M1_point is not None else None,
        'P1': np.asarray(grid.P1_point, dtype=float) if grid.P1_point is not None else None,
    }


def point_error(src, dst):
    if src is None or dst is None:
        return None
    return float(np.linalg.norm(np.asarray(src, dtype=float) - np.asarray(dst, dtype=float)))


def rms_point_error(src, dst):
    src = np.asarray(src, dtype=float)
    dst = np.asarray(dst, dtype=float)
    return float(np.sqrt(np.mean(np.sum((src - dst) ** 2, axis=1))))


def compute_true_metrics(mapped, target):
    metrics = {
        'spine_rms': rms_point_error(mapped['C'], target['C']),
        'M1_err': point_error(mapped.get('M1'), target.get('M1')),
        'L1_err': point_error(mapped['L1'], target['L1']),
        'L6_err': point_error(mapped['L6'], target['L6']),
        'P1_err': point_error(mapped.get('P1'), target.get('P1')),
    }
    metrics['L_2pt_rms'] = rms_point_error(
        np.vstack([mapped['L1'], mapped['L6']]),
        np.vstack([target['L1'], target['L6']]),
    )
    return metrics


lm_A = extract_true_landmarks(grid_A)
lm_B = extract_true_landmarks(grid_B)

print('Landmarks:')
for label, lm in [('Speaker A', lm_A), ('Speaker B', lm_B)]:
    print(f'  {label}:')
    print(f"    C points: {lm['C'].shape[0]}")
    print(f"    L1: {np.round(lm['L1'], 1)}")
    print(f"    L6: {np.round(lm['L6'], 1)}")
    print(f"    M1: {None if lm['M1'] is None else np.round(lm['M1'], 1)}")
    print(f"    P1: {None if lm['P1'] is None else np.round(lm['P1'], 1)}")

# ------------------------------------------------------------------
# Transform estimators (pure numpy, SVD-based)
# ------------------------------------------------------------------
def estimate_similarity(src, dst):
    """Similarity transform (rotation + isotropic scale + translation) via Procrustes/SVD."""
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    mu_s, mu_d = src.mean(0), dst.mean(0)
    S, D = src - mu_s, dst - mu_d
    H = S.T @ D
    U, Sigma, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign = np.array([1.0, np.sign(d)])
    R = (Vt.T * sign) @ U.T
    var_s = np.sum(S ** 2)
    s = float(np.sum(Sigma * sign) / var_s) if var_s > 1e-12 else 1.0
    t = mu_d - s * (R @ mu_s)
    return {'R': R, 's': s, 't': t, 'type': 'similarity'}


def estimate_affine(src, dst):
    """Full 2-D affine transform (2x2 A + translation t) via least-squares."""
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    N = len(src)
    M = np.zeros((2 * N, 6))
    b = np.zeros(2 * N)
    for i in range(N):
        x, y = src[i]; u, v = dst[i]
        M[2*i]   = [x, y, 1, 0, 0, 0]
        M[2*i+1] = [0, 0, 0, x, y, 1]
        b[2*i] = u; b[2*i+1] = v
    params, *_ = np.linalg.lstsq(M, b, rcond=None)
    A = np.array([[params[0], params[1]], [params[3], params[4]]])
    t = np.array([params[2], params[5]])
    return {'A': A, 't': t, 'type': 'affine'}


def apply_transform(T, pts):
    """Apply a Similarity or Affine transform to point(s).  pts: (N,2) or (2,)."""
    pts = np.asarray(pts, float)
    single = (pts.ndim == 1)
    if single:
        pts = pts.reshape(1, 2)
    if T['type'] == 'similarity':
        out = (T['s'] * (T['R'] @ pts.T)).T + T['t']
    else:
        out = (T['A'] @ pts.T).T + T['t']
    return out[0] if single else out


# ------------------------------------------------------------------
# Evaluate a transform on true landmarks
# ------------------------------------------------------------------
def evaluate_transform(T, lm_src, lm_tgt, label=""):
    """Apply T to true landmarks of src, compute metrics vs tgt."""
    res = {'label': label}

    # Map all true landmarks
    lm_hat = {}
    lm_hat['C']  = apply_transform(T, lm_src['C'])
    lm_hat['L1'] = apply_transform(T, lm_src['L1'])
    lm_hat['L6'] = apply_transform(T, lm_src['L6'])
    lm_hat['M1'] = apply_transform(T, lm_src['M1']) if lm_src['M1'] is not None else None
    lm_hat['P1'] = apply_transform(T, lm_src['P1']) if lm_src['P1'] is not None else None

    # Compute standard metrics
    res.update(compute_true_metrics(lm_hat, lm_tgt))

    # Transformed grid lines (for overlay visualisation)
    res['horiz'] = [apply_transform(T, h) for h in lm_src.get('_horiz', [])]
    res['vert']  = [apply_transform(T, v) for v in lm_src.get('_vert', [])]

    return res


# ------------------------------------------------------------------
# Fit transforms on spine C1..C6
# ------------------------------------------------------------------
T_sim = estimate_similarity(lm_B['C'], lm_A['C'])
T_aff = estimate_affine(lm_B['C'], lm_A['C'])

print("Similarity transform (fitted on spine C1..C6):")
print(f"  Scale     = {T_sim['s']:.4f}")
angle_deg = np.degrees(np.arctan2(T_sim['R'][1, 0], T_sim['R'][0, 0]))
print(f"  Rotation  = {angle_deg:.2f}°")
print(f"  Translate = ({T_sim['t'][0]:.1f}, {T_sim['t'][1]:.1f})")
print(f"\nAffine transform (fitted on spine C1..C6):")
print(f"  A = {T_aff['A'].round(4).tolist()}")
print(f"  t = ({T_aff['t'][0]:.1f}, {T_aff['t'][1]:.1f})")

# Store raw grid lines in lm dicts
lm_A['_horiz'] = [h.copy() for h in grid_A.horiz_lines]
lm_A['_vert']  = [v.copy() for v in grid_A.vert_lines]
lm_B['_horiz'] = [h.copy() for h in grid_B.horiz_lines]
lm_B['_vert']  = [v.copy() for v in grid_B.vert_lines]

res_sim = evaluate_transform(T_sim, lm_B, lm_A, "Similarity (spine)")
res_aff = evaluate_transform(T_aff, lm_B, lm_A, "Affine (spine)")


# ------------------------------------------------------------------
# Results table  (true landmarks only)
# ------------------------------------------------------------------
METRIC_KEYS = [
    ('spine_rms',  'Spine C1..C6   RMS'),
    ('M1_err',     'M1              err'),
    ('L1_err',     'L1              err'),
    ('L6_err',     'L6              err'),
    ('L_2pt_rms',  'L (L1,L6)  2pt RMS  ★'),
    ('P1_err',     'P1              err'),
]

print("\n" + "=" * 70)
print(f"{'METRIC (px)':34s} {'Similarity':>14s} {'Affine':>14s}")
print("=" * 70)
for key, nice in METRIC_KEYS:
    v1 = res_sim.get(key); v2 = res_aff.get(key)
    s1 = f"{v1:.2f}" if v1 is not None else "—"
    s2 = f"{v2:.2f}" if v2 is not None else "—"
    print(f"  {nice:32s} {s1:>14s} {s2:>14s}")
print("=" * 70)


# ------------------------------------------------------------------
# Visualise grid overlay
# ------------------------------------------------------------------
def draw_grid_lines(ax, horiz, vert, color_h='c', color_v='y', alpha=0.6, lw=1.0):
    for h in horiz:
        ax.plot(h[:, 0], h[:, 1], color=color_h, lw=lw, alpha=alpha)
    for v in vert:
        ax.plot(v[:, 0], v[:, 1], color=color_v, lw=lw, alpha=alpha)


fig, axes = plt.subplots(1, 3, figsize=(36, 12))
h_img, w_img = np.asarray(image_A).shape[:2]

for ax_i, (ax, res, title) in enumerate(zip(axes, [None, res_sim, res_aff],
    ["Before alignment\ncyan/yellow=A, red/orange=B",
     f"Similarity (spine)\nL_2pt_RMS = {res_sim['L_2pt_rms']:.1f} px",
     f"Affine (spine)\nL_2pt_RMS = {res_aff['L_2pt_rms']:.1f} px"])):
    ax.imshow(image_A, cmap='gray')
    draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
    if res is None:
        draw_grid_lines(ax, lm_B['_horiz'], lm_B['_vert'], color_h='red', color_v='orange', alpha=0.5, lw=1.2)
    else:
        draw_grid_lines(ax, res['horiz'], res['vert'], color_h='red', color_v='orange', alpha=0.6, lw=1.2)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  PART 3 — Method 2: Curvilinear spine-based transfer  (u, r)
# =====================================================================
from scipy.interpolate import interp1d

# ------------------------------------------------------------------
# Build arc-length parameterised spine polylines
# ------------------------------------------------------------------
def build_spine_polyline(grid, n_resample=200):
    """
    Resample spine polyline C1..C6 to n_resample equi-arc-length points.
    Returns spine (n,2) and u_vals (n,) normalised arc-length [0,1].
    """
    cc = grid.cervical_centers
    pts = np.array([cc[f'c{i}'] for i in range(1, 7)], dtype=float)
    diffs = np.diff(pts, axis=0)
    seg_len = np.linalg.norm(diffs, axis=1)
    arc = np.concatenate([[0], np.cumsum(seg_len)])
    arc_norm = arc / arc[-1]
    u_new = np.linspace(0, 1, n_resample)
    fx = interp1d(arc_norm, pts[:, 0], kind='linear')
    fy = interp1d(arc_norm, pts[:, 1], kind='linear')
    return np.column_stack([fx(u_new), fy(u_new)]), u_new


# ------------------------------------------------------------------
#  (u, r)  projection & reconstruction
# ------------------------------------------------------------------
def project_onto_spine(pts, spine, u_vals):
    """For each point in pts compute (u, r). pts can be (N,2) or (2,)."""
    single = (np.asarray(pts).ndim == 1)
    pts = np.atleast_2d(pts)
    N, M = len(pts), len(spine)
    u_out, r_out = np.empty(N), np.empty(N)
    for i in range(N):
        p = pts[i]
        dists = np.linalg.norm(spine - p, axis=1)
        j = int(np.argmin(dists))
        u_out[i] = u_vals[j]
        if j == 0:       tang = spine[1] - spine[0]
        elif j == M - 1: tang = spine[-1] - spine[-2]
        else:            tang = spine[j+1] - spine[j-1]
        tang /= (np.linalg.norm(tang) + 1e-12)
        normal = np.array([tang[1], -tang[0]])
        r_out[i] = np.dot(p - spine[j], normal)
    return (u_out[0], r_out[0]) if single else (u_out, r_out)


def reconstruct_from_spine(u_arr, r_arr, spine, u_vals):
    """Reconstruct points from (u,r) in a target spine frame."""
    single = np.isscalar(u_arr)
    u_arr = np.atleast_1d(u_arr); r_arr = np.atleast_1d(r_arr)
    N = len(u_arr)
    pts_hat = np.empty((N, 2))
    fx = interp1d(u_vals, spine[:, 0], kind='linear',
                  bounds_error=False, fill_value='extrapolate')
    fy = interp1d(u_vals, spine[:, 1], kind='linear',
                  bounds_error=False, fill_value='extrapolate')
    du = 1e-5
    for i in range(N):
        u = u_arr[i]
        sx, sy = float(fx(u)), float(fy(u))
        tx = float(fx(u+du)) - float(fx(u-du))
        ty = float(fy(u+du)) - float(fy(u-du))
        tn = np.sqrt(tx**2 + ty**2) + 1e-12
        tx /= tn; ty /= tn
        nx, ny = ty, -tx
        pts_hat[i] = [sx + r_arr[i]*nx, sy + r_arr[i]*ny]
    return pts_hat[0] if single else pts_hat


# ------------------------------------------------------------------
# Apply Method 2 to true landmarks + grid
# ------------------------------------------------------------------
spine_A, u_A = build_spine_polyline(grid_A, n_resample=200)
spine_B, u_B = build_spine_polyline(grid_B, n_resample=200)

# Map true landmarks through (u,r) transfer
mapped_ur = {}

# C1..C6
u_c, r_c = project_onto_spine(lm_B['C'], spine_B, u_B)
mapped_ur['C'] = reconstruct_from_spine(u_c, r_c, spine_A, u_A)

# L1, L6
for lk in ('L1', 'L6'):
    u_, r_ = project_onto_spine(lm_B[lk], spine_B, u_B)
    mapped_ur[lk] = reconstruct_from_spine(u_, r_, spine_A, u_A)

# M1
if lm_B['M1'] is not None:
    u_, r_ = project_onto_spine(lm_B['M1'], spine_B, u_B)
    mapped_ur['M1'] = reconstruct_from_spine(u_, r_, spine_A, u_A)
else:
    mapped_ur['M1'] = None

# P1
if lm_B['P1'] is not None:
    u_, r_ = project_onto_spine(lm_B['P1'], spine_B, u_B)
    mapped_ur['P1'] = reconstruct_from_spine(u_, r_, spine_A, u_A)
else:
    mapped_ur['P1'] = None

# Map full grid lines for visualisation
horiz_ur = []
for h in grid_B.horiz_lines:
    u_, r_ = project_onto_spine(h, spine_B, u_B)
    horiz_ur.append(reconstruct_from_spine(u_, r_, spine_A, u_A))
vert_ur = []
for v in grid_B.vert_lines:
    u_, r_ = project_onto_spine(v, spine_B, u_B)
    vert_ur.append(reconstruct_from_spine(u_, r_, spine_A, u_A))


# ------------------------------------------------------------------
# Metrics  (true landmarks)
# ------------------------------------------------------------------
res_ur = {'label': 'Curvilinear (u,r)'}
res_ur.update(compute_true_metrics(mapped_ur, lm_A))
res_ur['horiz'] = horiz_ur
res_ur['vert']  = vert_ur

print("=" * 70)
print("METHOD 2  —  Curvilinear spine-based transfer  (u, r)")
print("=" * 70)
for key, nice in METRIC_KEYS:
    v = res_ur.get(key)
    s = f"{v:.2f}" if v is not None else "—"
    print(f"  {nice:32s} {s:>12s}")
print("=" * 70)


# ------------------------------------------------------------------
# Visualise
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(36, 12))
h_img, w_img = np.asarray(image_A).shape[:2]

# Panel 1: Before
ax = axes[0]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
draw_grid_lines(ax, lm_B['_horiz'], lm_B['_vert'], color_h='red', color_v='orange', alpha=0.5, lw=1.2)
ax.set_title("Before alignment\ncyan/yellow=A, red/orange=B", fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 2: Method 2 grid overlay
ax = axes[1]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
draw_grid_lines(ax, horiz_ur, vert_ur, color_h='red', color_v='orange', alpha=0.6, lw=1.2)
ax.set_title(f"Method 2: Curvilinear (u,r)\nL_2pt_RMS = {res_ur['L_2pt_rms']:.1f} px",
             fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 3: mapped true landmarks vs targets
ax = axes[2]
ax.imshow(image_A, cmap='gray', alpha=0.5)
# A true landmarks
ax.plot(lm_A['C'][:, 0], lm_A['C'][:, 1], 'bs', ms=10, mec='white', mew=1.5, label='A  C')
ax.plot(*lm_A['L1'], 'bD', ms=11, mec='white', mew=1.5, label='A  L1')
ax.plot(*lm_A['L6'], 'bv', ms=11, mec='white', mew=1.5, label='A  L6')
if lm_A['M1'] is not None: ax.plot(*lm_A['M1'], 'b^', ms=11, mec='white', mew=1.5, label='A  M1')
if lm_A['P1'] is not None: ax.plot(*lm_A['P1'], 'bp', ms=11, mec='white', mew=1.5, label='A  P1')
# Mapped B
ax.plot(mapped_ur['C'][:, 0], mapped_ur['C'][:, 1], 'rs', ms=10, mec='white', mew=1.5, label='B→A C')
ax.plot(*mapped_ur['L1'], 'rD', ms=11, mec='white', mew=1.5, label='B→A L1')
ax.plot(*mapped_ur['L6'], 'rv', ms=11, mec='white', mew=1.5, label='B→A L6')
if mapped_ur['M1'] is not None: ax.plot(*mapped_ur['M1'], 'r^', ms=11, mec='white', mew=1.5, label='B→A M1')
if mapped_ur['P1'] is not None: ax.plot(*mapped_ur['P1'], 'rp', ms=11, mec='white', mew=1.5, label='B→A P1')
# Residual arrows
for i in range(len(lm_A['C'])):
    ax.annotate('', xy=lm_A['C'][i], xytext=mapped_ur['C'][i],
                arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
for lk in ('L1', 'L6', 'M1', 'P1'):
    if mapped_ur.get(lk) is not None and lm_A.get(lk) is not None:
        ax.annotate('', xy=lm_A[lk], xytext=mapped_ur[lk],
                    arrowprops=dict(arrowstyle='->', color='lime', lw=2))
ax.set_title("Method 2: Mapped true landmarks\ngreen arrows = residual",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', ncol=2)
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  PART 4 — Method 3: TPS warping  (fit C+P1, test M1/L1/L6)
# =====================================================================
from scipy.interpolate import RBFInterpolator

# ------------------------------------------------------------------
# TPS helpers
# ------------------------------------------------------------------
def fit_tps(src, dst, smoothing=0.0):
    """Fit 2D TPS: src → dst via RBFInterpolator."""
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    rbf_x = RBFInterpolator(src, dst[:, 0], kernel='thin_plate_spline', smoothing=smoothing)
    rbf_y = RBFInterpolator(src, dst[:, 1], kernel='thin_plate_spline', smoothing=smoothing)
    return {'rbf_x': rbf_x, 'rbf_y': rbf_y, 'type': 'tps'}


def apply_tps(tps, pts):
    """Apply fitted TPS. pts: (N,2) or (2,)."""
    pts = np.asarray(pts, float)
    single = (pts.ndim == 1)
    if single:
        pts = pts.reshape(1, 2)
    x_hat = tps['rbf_x'](pts)
    y_hat = tps['rbf_y'](pts)
    out = np.column_stack([x_hat, y_hat])
    return out[0] if single else out


# ------------------------------------------------------------------
# Build control & test sets from true landmarks
# ------------------------------------------------------------------
# Control (fit):  C1..C6  +  P1 (if available)
ctrl_src = [lm_B['C'].copy()]
ctrl_tgt = [lm_A['C'].copy()]
ctrl_labels = [f'C{i+1}' for i in range(len(lm_B['C']))]

if lm_B['P1'] is not None and lm_A['P1'] is not None:
    ctrl_src.append(lm_B['P1'].reshape(1, 2))
    ctrl_tgt.append(lm_A['P1'].reshape(1, 2))
    ctrl_labels.append('P1')

ctrl_src = np.vstack(ctrl_src)   # (7, 2) if P1 present, else (6, 2)
ctrl_tgt = np.vstack(ctrl_tgt)

# Test (evaluate):  M1, L1, L6
test_names = []
test_src_list = []
test_tgt_list = []
for name in ('M1', 'L1', 'L6'):
    if lm_B.get(name) is not None and lm_A.get(name) is not None:
        test_names.append(name)
        test_src_list.append(np.asarray(lm_B[name], float))
        test_tgt_list.append(np.asarray(lm_A[name], float))
test_src = np.array(test_src_list)   # (3, 2)
test_tgt = np.array(test_tgt_list)

print("=" * 70)
print("METHOD 3  —  TPS  (control: C1..C6 + P1, test: M1, L1, L6)")
print("=" * 70)
print(f"  Control points : {len(ctrl_src)}  ({', '.join(ctrl_labels)})")
print(f"  Test points    : {len(test_src)}  ({', '.join(test_names)})")


# ------------------------------------------------------------------
# Fit TPS and evaluate
# ------------------------------------------------------------------
tps_model = fit_tps(ctrl_src, ctrl_tgt, smoothing=0.0)

# Control-set residuals (fit quality — should be ~0)
ctrl_hat = apply_tps(tps_model, ctrl_src)
ctrl_rms = rms_point_error(ctrl_hat, ctrl_tgt)

# Test-set evaluation
test_hat = apply_tps(tps_model, test_src)
test_errors = {}
for i, name in enumerate(test_names):
    test_errors[name] = point_error(test_hat[i], test_tgt[i])

test_rms  = rms_point_error(test_hat, test_tgt)
test_mean = float(np.mean(list(test_errors.values())))

print(f"\n  --- Control residuals ---")
print(f"  spine_rms (fit)  = {ctrl_rms:.4f} px")
print(f"\n  --- Test errors ---")
for name, err in test_errors.items():
    print(f"  {name:4s}  err = {err:.2f} px")
print(f"  test_rms         = {test_rms:.2f} px")
print(f"  test_mean        = {test_mean:.2f} px")

# Build res_tps dict consistent with other methods
res_tps = {
    'label': 'TPS (C+P1)',
    'spine_rms':  ctrl_rms,
    'M1_err':     test_errors.get('M1'),
    'L1_err':     test_errors.get('L1'),
    'L6_err':     test_errors.get('L6'),
    'L_2pt_rms':  rms_point_error(
        np.array([apply_tps(tps_model, lm_B['L1']), apply_tps(tps_model, lm_B['L6'])]),
        np.array([lm_A['L1'], lm_A['L6']])),
    'P1_err':     point_error(apply_tps(tps_model, lm_B['P1']), lm_A['P1']) if (
        lm_B['P1'] is not None and lm_A['P1'] is not None) else None,
    'test_rms':   test_rms,
    'test_mean':  test_mean,
}


# ------------------------------------------------------------------
# Warp grid of B through TPS
# ------------------------------------------------------------------
horiz_tps = [apply_tps(tps_model, h) for h in grid_B.horiz_lines]
vert_tps  = [apply_tps(tps_model, v) for v in grid_B.vert_lines]
res_tps['horiz'] = horiz_tps
res_tps['vert']  = vert_tps


# ------------------------------------------------------------------
# Results table
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("METHOD 3  —  TPS results  (true landmarks)")
print("=" * 70)
for key, nice in METRIC_KEYS:
    v = res_tps.get(key)
    s = f"{v:.2f}" if v is not None else "—"
    note = " (fit)" if key == 'spine_rms' else ""
    note = " (fit)" if key == 'P1_err' else note
    print(f"  {nice:32s} {s:>12s}{note}")
print(f"  {'Test RMS  (M1,L1,L6)':32s} {res_tps['test_rms']:>12.2f}  ★")
print(f"  {'Test Mean (M1,L1,L6)':32s} {res_tps['test_mean']:>12.2f}")
print("=" * 70)


# ------------------------------------------------------------------
# Visualise
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(36, 12))
h_img, w_img = np.asarray(image_A).shape[:2]

# Panel 1: Before alignment
ax = axes[0]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
draw_grid_lines(ax, lm_B['_horiz'], lm_B['_vert'], color_h='red', color_v='orange', alpha=0.5, lw=1.2)
ax.set_title("Before alignment\ncyan/yellow=A, red/orange=B", fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 2: TPS grid overlay
ax = axes[1]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
draw_grid_lines(ax, horiz_tps, vert_tps, color_h='red', color_v='orange', alpha=0.6, lw=1.2)
# Mark control & test points
ax.plot(ctrl_tgt[:, 0], ctrl_tgt[:, 1], 'g^', ms=12, mec='white', mew=1.5,
        label=f'ctrl ({len(ctrl_tgt)} pts)', zorder=5)
ax.plot(test_tgt[:, 0], test_tgt[:, 1], 'm*', ms=14, mec='white', mew=1.5,
        label=f'test ({len(test_tgt)} pts)', zorder=5)
for i, nm in enumerate(test_names):
    ax.annotate(nm, test_tgt[i], fontsize=10, color='magenta', fontweight='bold',
                xytext=(6, -14), textcoords='offset points')
ax.legend(fontsize=11, loc='upper right')
ax.set_title(f"Method 3: TPS ({len(ctrl_src)} ctrl)\ntest_rms = {test_rms:.1f} px",
             fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 3: per-point test error bar chart + mapped points
ax = axes[2]
colors_bar = ['#e74c3c' if e == max(test_errors.values()) else '#3498db'
              for e in test_errors.values()]
bars = ax.barh(test_names, list(test_errors.values()), color=colors_bar,
               edgecolor='navy', height=0.5)
for i, (nm, e) in enumerate(test_errors.items()):
    ax.text(e + 0.5, i, f"{e:.1f} px", va='center', fontsize=13, fontweight='bold')
ax.axvline(test_rms, color='green', ls='--', lw=2, label=f'test_rms = {test_rms:.1f}')
ax.axvline(test_mean, color='orange', ls=':', lw=2, label=f'test_mean = {test_mean:.1f}')
ax.set_xlabel("Error (px)", fontsize=13)
ax.set_title("TPS — per-point test errors\n(M1, L1, L6 not used in fit)",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  PART 4b — Method 4: Two-step alignment
#     Step 1: Affine transform fitted on C1..C6 + L1 + L6
#             (makes spine AND lateral endpoints as close as possible)
#     Step 2: TPS warp on residuals for remaining points (M1, P1, grid)
# =====================================================================

# ------------------------------------------------------------------
# Step 1: Affine fitted on C1..C6 + L1 + L6  (8 anchor points)
# ------------------------------------------------------------------
anchor_src = np.vstack([
    lm_B['C'],                       # C1..C6  (6 pts)
    lm_B['L1'].reshape(1, 2),        # L1
    lm_B['L6'].reshape(1, 2),        # L6
])
anchor_tgt = np.vstack([
    lm_A['C'],
    lm_A['L1'].reshape(1, 2),
    lm_A['L6'].reshape(1, 2),
])
anchor_labels = [f'C{i+1}' for i in range(6)] + ['L1', 'L6']

T_step1 = estimate_affine(anchor_src, anchor_tgt)

print("=" * 70)
print("METHOD 4  —  Two-step alignment")
print("=" * 70)
print(f"\nStep 1: Affine fitted on {len(anchor_src)} anchors ({', '.join(anchor_labels)})")
print(f"  A = {T_step1['A'].round(4).tolist()}")
print(f"  t = ({T_step1['t'][0]:.1f}, {T_step1['t'][1]:.1f})")

# Apply Step 1 to ALL B landmarks
step1_C  = apply_transform(T_step1, lm_B['C'])
step1_L1 = apply_transform(T_step1, lm_B['L1'])
step1_L6 = apply_transform(T_step1, lm_B['L6'])
step1_M1 = apply_transform(T_step1, lm_B['M1']) if lm_B['M1'] is not None else None
step1_P1 = apply_transform(T_step1, lm_B['P1']) if lm_B['P1'] is not None else None

# Step 1 residuals on anchors
step1_anchor_rms = rms_point_error(apply_transform(T_step1, anchor_src), anchor_tgt)
print(f"  Anchor RMS (C1..C6+L1+L6) = {step1_anchor_rms:.2f} px")

# Step 1 residuals on other points
if step1_M1 is not None and lm_A['M1'] is not None:
    print(f"  M1 after step1 = {point_error(step1_M1, lm_A['M1']):.2f} px")
if step1_P1 is not None and lm_A['P1'] is not None:
    print(f"  P1 after step1 = {point_error(step1_P1, lm_A['P1']):.2f} px")


# ------------------------------------------------------------------
# Step 2: TPS on residuals (fine-tune non-anchor points)
# ------------------------------------------------------------------
# After Step 1, the 8 anchor points are already close to targets.
# We use ALL available landmarks after Step 1 as TPS control points
# to warp the remaining grid points.

tps2_src = [step1_C.copy()]
tps2_tgt = [lm_A['C'].copy()]
tps2_labels = [f'C{i+1}' for i in range(6)]

tps2_src.append(step1_L1.reshape(1, 2))
tps2_tgt.append(lm_A['L1'].reshape(1, 2))
tps2_labels.append('L1')

tps2_src.append(step1_L6.reshape(1, 2))
tps2_tgt.append(lm_A['L6'].reshape(1, 2))
tps2_labels.append('L6')

if step1_P1 is not None and lm_A['P1'] is not None:
    tps2_src.append(step1_P1.reshape(1, 2))
    tps2_tgt.append(lm_A['P1'].reshape(1, 2))
    tps2_labels.append('P1')

tps2_src = np.vstack(tps2_src)
tps2_tgt = np.vstack(tps2_tgt)

tps_step2 = fit_tps(tps2_src, tps2_tgt, smoothing=0.0)

print(f"\nStep 2: TPS refinement with {len(tps2_src)} control points ({', '.join(tps2_labels)})")

# Full pipeline: apply Step 1 (affine) then Step 2 (TPS)
def apply_two_step(pts):
    """Apply Step 1 affine then Step 2 TPS to point(s)."""
    pts = np.asarray(pts, float)
    single = (pts.ndim == 1)
    if single:
        pts = pts.reshape(1, 2)
    intermediate = apply_transform(T_step1, pts)
    result = apply_tps(tps_step2, intermediate)
    return result[0] if single else result


# Map true landmarks through two-step pipeline
mapped_2s = {}
mapped_2s['C']  = apply_two_step(lm_B['C'])
mapped_2s['L1'] = apply_two_step(lm_B['L1'])
mapped_2s['L6'] = apply_two_step(lm_B['L6'])
mapped_2s['M1'] = apply_two_step(lm_B['M1']) if lm_B['M1'] is not None else None
mapped_2s['P1'] = apply_two_step(lm_B['P1']) if lm_B['P1'] is not None else None


# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------
res_2step = {'label': 'Two-step (Affine+TPS)'}
res_2step.update(compute_true_metrics(mapped_2s, lm_A))

# Test metrics for M1 (not directly constrained in step 1)
test_2s_names, test_2s_errs = [], []
if mapped_2s['M1'] is not None and lm_A['M1'] is not None:
    test_2s_names.append('M1')
    test_2s_errs.append(point_error(mapped_2s['M1'], lm_A['M1']))

if len(test_2s_errs) > 0:
    res_2step['test_rms'] = float(np.sqrt(np.mean(np.array(test_2s_errs)**2)))
    res_2step['test_mean'] = float(np.mean(test_2s_errs))
else:
    res_2step['test_rms'] = res_2step.get('L_2pt_rms')
    res_2step['test_mean'] = res_2step.get('L_2pt_rms')

# Warp full grid through two-step pipeline
horiz_2s = [apply_two_step(h) for h in grid_B.horiz_lines]
vert_2s  = [apply_two_step(v) for v in grid_B.vert_lines]
res_2step['horiz'] = horiz_2s
res_2step['vert']  = vert_2s

print("\n" + "=" * 70)
print("METHOD 4  —  Two-step results  (true landmarks)")
print("=" * 70)
for key, nice in METRIC_KEYS:
    v = res_2step.get(key)
    s = f"{v:.2f}" if v is not None else "—"
    print(f"  {nice:32s} {s:>12s}")
print("=" * 70)


# ------------------------------------------------------------------
# Visualise two-step alignment
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(36, 12))
h_img, w_img = np.asarray(image_A).shape[:2]

# Panel 1: After Step 1 only (affine on C+L)
ax = axes[0]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
step1_horiz = [apply_transform(T_step1, h) for h in grid_B.horiz_lines]
step1_vert  = [apply_transform(T_step1, v) for v in grid_B.vert_lines]
draw_grid_lines(ax, step1_horiz, step1_vert, color_h='red', color_v='orange', alpha=0.6, lw=1.2)
ax.set_title(f"Step 1: Affine (C1..C6+L1+L6)\nAnchor RMS = {step1_anchor_rms:.1f} px",
             fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 2: After Step 2 (TPS refinement)
ax = axes[1]
ax.imshow(image_A, cmap='gray')
draw_grid_lines(ax, lm_A['_horiz'], lm_A['_vert'], color_h='cyan', color_v='yellow', lw=1.5)
draw_grid_lines(ax, horiz_2s, vert_2s, color_h='red', color_v='orange', alpha=0.6, lw=1.2)
ax.set_title(f"Step 2: + TPS refinement\nL_2pt_RMS = {res_2step['L_2pt_rms']:.1f} px",
             fontsize=14, fontweight='bold')
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

# Panel 3: Mapped landmarks with residual arrows
ax = axes[2]
ax.imshow(image_A, cmap='gray', alpha=0.5)
# A true landmarks
ax.plot(lm_A['C'][:, 0], lm_A['C'][:, 1], 'bs', ms=10, mec='white', mew=1.5, label='A  C')
ax.plot(*lm_A['L1'], 'bD', ms=11, mec='white', mew=1.5, label='A  L1')
ax.plot(*lm_A['L6'], 'bv', ms=11, mec='white', mew=1.5, label='A  L6')
if lm_A['M1'] is not None: ax.plot(*lm_A['M1'], 'b^', ms=11, mec='white', mew=1.5, label='A  M1')
if lm_A['P1'] is not None: ax.plot(*lm_A['P1'], 'bp', ms=11, mec='white', mew=1.5, label='A  P1')
# Mapped B (two-step)
ax.plot(mapped_2s['C'][:, 0], mapped_2s['C'][:, 1], 'rs', ms=10, mec='white', mew=1.5, label='B→A C')
ax.plot(*mapped_2s['L1'], 'rD', ms=11, mec='white', mew=1.5, label='B→A L1')
ax.plot(*mapped_2s['L6'], 'rv', ms=11, mec='white', mew=1.5, label='B→A L6')
if mapped_2s['M1'] is not None: ax.plot(*mapped_2s['M1'], 'r^', ms=11, mec='white', mew=1.5, label='B→A M1')
if mapped_2s['P1'] is not None: ax.plot(*mapped_2s['P1'], 'rp', ms=11, mec='white', mew=1.5, label='B→A P1')
# Residual arrows
for i in range(len(lm_A['C'])):
    ax.annotate('', xy=lm_A['C'][i], xytext=mapped_2s['C'][i],
                arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
for lk in ('L1', 'L6', 'M1', 'P1'):
    if mapped_2s.get(lk) is not None and lm_A.get(lk) is not None:
        ax.annotate('', xy=lm_A[lk], xytext=mapped_2s[lk],
                    arrowprops=dict(arrowstyle='->', color='lime', lw=2))
ax.set_title("Two-step: Mapped true landmarks\ngreen arrows = residual",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', ncol=2)
ax.axis('off'); ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)

plt.tight_layout()
plt.show()

In [ ]:
# 5 — Summary comparison  (true landmarks only)

import pandas as pd

def _fmt(v):
    return f"{v:.2f}" if v is not None else "—"

rows = []
for res, method in [
    (res_sim,   'M1 Similarity (spine)'),
    (res_aff,   'M1 Affine (spine)'),
    (res_ur,    'M2 Curvilinear (u,r)'),
    (res_tps,   'M3 TPS (C+P1)'),
    (res_2step, 'M4 Two-step (Affine+TPS)'),
]:
    rows.append({
        'Method':     method,
        'spine_rms':  res['spine_rms'],
        'M1_err':     res.get('M1_err'),
        'L1_err':     res.get('L1_err'),
        'L6_err':     res.get('L6_err'),
        'L_2pt_rms':  res.get('L_2pt_rms'),
        'P1_err':     res.get('P1_err'),
        # Main generalisation metric:
        #   M1/M2: L_2pt_rms  (L1,L6 are test points — not used to fit)
        #   TPS:   test_rms   (M1,L1,L6 not used to fit)
        'main_metric': res.get('test_rms', res.get('L_2pt_rms')),
    })

df = pd.DataFrame(rows).sort_values('main_metric').reset_index(drop=True)

# Display
cols = ['Method', 'spine_rms', 'M1_err', 'L1_err', 'L6_err', 'L_2pt_rms', 'P1_err', 'main_metric']
df_display = df[cols].copy()
for c in cols[1:]:
    df_display[c] = df_display[c].map(_fmt)

print("=" * 105)
print("SUMMARY  —  All methods ranked by main generalisation metric  (px)")
print("=" * 105)
print(df_display.to_string(index=False))
print("=" * 105)
print("main_metric = L_2pt_rms for M1/M2  (L1,L6 not in fit)")
print("              test_rms  for TPS    (M1,L1,L6 not in fit)")
print()


# ------------------------------------------------------------------
# Bar plot of main metric
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))

methods = df['Method'].tolist()
values  = df['main_metric'].tolist()

colors = ['#2ecc71' if v == min(values) else '#3498db' for v in values]
bars = ax.barh(range(len(methods)), values, color=colors, edgecolor='navy', height=0.55)

for i, v in enumerate(values):
    ax.text(v + max(values) * 0.02, i, f"{v:.1f}", va='center',
            fontsize=12, fontweight='bold')

ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods, fontsize=12)
ax.set_xlabel("Main generalisation metric (px) — lower is better", fontsize=13)
ax.set_title("Cross-speaker grid alignment — Method comparison\n(true landmarks only)",
             fontsize=15, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(0, max(values) * 1.25)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Detailed per-landmark bar chart
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

lm_names = ['M1_err', 'L1_err', 'L6_err', 'P1_err']
lm_labels = ['M1', 'L1', 'L6', 'P1']
x = np.arange(len(lm_labels))
width = 0.15

method_colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']
method_names = [r['Method'] for _, r in df.iterrows()]

for i, (_, row) in enumerate(df.iterrows()):
    vals = [row[k] if row[k] is not None else 0 for k in lm_names]
    ax.bar(x + i * width, vals, width, label=row['Method'],
           color=method_colors[i], edgecolor='navy', alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(lm_labels, fontsize=13)
ax.set_ylabel("Error (px)", fontsize=13)
ax.set_title("Per-landmark errors by method", fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
#  Simple visual helpers for two-speaker overlays
# =====================================================================
from matplotlib.lines import Line2D

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_GRID_COLOR = '#16a085'
SOURCE_GRID_COLOR = '#d97706'

h_img, w_img = np.asarray(image_A).shape[:2]

def draw_grid_single_color(ax, horiz, vert, color, *, alpha=0.95,
                           lw_major=3.0, lw_minor=1.0):
    """Draw one speaker grid using one color only.
    Thick lines mark the outer grid; thin lines mark the interior.
    """
    for i, h in enumerate(horiz):
        lw = lw_major if i in (0, len(horiz) - 1) else lw_minor
        ax.plot(h[:, 0], h[:, 1], '-', color=color, lw=lw, alpha=alpha)
    for j, v in enumerate(vert):
        lw = lw_major if j in (0, len(vert) - 1) else lw_minor
        ax.plot(v[:, 0], v[:, 1], '-', color=color, lw=lw, alpha=alpha)


def format_target_frame(ax, title):
    ax.imshow(image_A, cmap='gray')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    ax.set_xlim(0, w_img)
    ax.set_ylim(h_img, 0)


def draw_named_points(ax, pts, labels, color, marker='o', ms=8, dx=6, dy=-6):
    for pt, lb in zip(pts, labels):
        ax.plot(pt[0], pt[1], marker=marker, ms=ms, color=color,
                mec='white', mew=1.2, zorder=8)
        ax.text(pt[0] + dx, pt[1] + dy, lb, fontsize=9, color=color,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                fc='white', alpha=0.8), zorder=9)


def speaker_legend(ax):
    handles = [
        Line2D([0], [0], color=TARGET_GRID_COLOR, lw=3, label='Target speaker grid'),
        Line2D([0], [0], color=SOURCE_GRID_COLOR, lw=3, label='Source speaker grid'),
    ]
    ax.legend(handles=handles, fontsize=10, loc='upper right', framealpha=0.92)


In [ ]:
# =====================================================================
#  Two grids only on one frame
# =====================================================================
fig, ax = plt.subplots(figsize=(12, 12))
format_target_frame(ax,
    'Two grids only on the target frame\n'
    'one color per speaker, line thickness = outer vs inner grid')

draw_grid_single_color(ax, lm_A['_horiz'], lm_A['_vert'], TARGET_GRID_COLOR,
                       alpha=0.95, lw_major=3.2, lw_minor=1.2)
draw_grid_single_color(ax, horiz_2s, vert_2s, SOURCE_GRID_COLOR,
                       alpha=0.88, lw_major=2.8, lw_minor=0.9)
speaker_legend(ax)

grid_only_path = OUTPUT_DIR / 'method4_two_grids_only.png'
fig.savefig(grid_only_path, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved: {grid_only_path}')


In [ ]:
# =====================================================================
#  Method 4 explained step by step
# =====================================================================
step1_anchor_hat = apply_transform(T_step1, anchor_src)
step2_ctrl_hat = apply_tps(tps_step2, tps2_src)

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.ravel()

# Panel 1: Before alignment
ax = axes[0]
format_target_frame(ax, 'Step 0: before alignment')
draw_grid_single_color(ax, lm_A['_horiz'], lm_A['_vert'], TARGET_GRID_COLOR,
                       alpha=0.95, lw_major=3.0, lw_minor=1.1)
draw_grid_single_color(ax, lm_B['_horiz'], lm_B['_vert'], SOURCE_GRID_COLOR,
                       alpha=0.70, lw_major=2.6, lw_minor=0.8)
speaker_legend(ax)

# Panel 2: Step 1 anchor fit
ax = axes[1]
format_target_frame(ax, 'Step 1a: affine fit on 8 anchors\n(C1..C6 + L1 + L6)')
draw_named_points(ax, anchor_tgt, anchor_labels, TARGET_GRID_COLOR, marker='o', ms=8)
draw_named_points(ax, step1_anchor_hat, anchor_labels, SOURCE_GRID_COLOR, marker='s', ms=7,
                  dx=6, dy=10)
for src_pt, tgt_pt in zip(step1_anchor_hat, anchor_tgt):
    ax.annotate('', xy=tgt_pt, xytext=src_pt,
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2, alpha=0.8))
ax.text(0.02, 0.03, f'Anchor RMS after affine = {step1_anchor_rms:.2f} px',
        transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9))

# Panel 3: Step 1 grid result
ax = axes[2]
format_target_frame(ax, 'Step 1b: target grid + affine-mapped source grid')
draw_grid_single_color(ax, lm_A['_horiz'], lm_A['_vert'], TARGET_GRID_COLOR,
                       alpha=0.95, lw_major=3.0, lw_minor=1.1)
draw_grid_single_color(ax, step1_horiz, step1_vert, SOURCE_GRID_COLOR,
                       alpha=0.85, lw_major=2.6, lw_minor=0.8)
speaker_legend(ax)

# Panel 4: Step 2 TPS controls
ax = axes[3]
format_target_frame(ax, 'Step 2a: TPS control points after Step 1')
draw_named_points(ax, tps2_tgt, tps2_labels, TARGET_GRID_COLOR, marker='o', ms=8)
draw_named_points(ax, tps2_src, tps2_labels, SOURCE_GRID_COLOR, marker='s', ms=7,
                  dx=6, dy=10)
for src_pt, tgt_pt in zip(tps2_src, tps2_tgt):
    ax.annotate('', xy=tgt_pt, xytext=src_pt,
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2, alpha=0.8))
ax.text(0.02, 0.03, 'TPS now bends the affine result so these controls land on the target.',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9))

# Panel 5: Final result
ax = axes[4]
format_target_frame(ax, 'Step 2b: final two-step result')
draw_grid_single_color(ax, lm_A['_horiz'], lm_A['_vert'], TARGET_GRID_COLOR,
                       alpha=0.95, lw_major=3.0, lw_minor=1.1)
draw_grid_single_color(ax, horiz_2s, vert_2s, SOURCE_GRID_COLOR,
                       alpha=0.88, lw_major=2.8, lw_minor=0.9)
speaker_legend(ax)

# Panel 6: Short metric summary
ax = axes[5]
ax.axis('off')
summary_lines = [
    'Method 4 summary',
    '',
    f'Step 1 anchor RMS: {step1_anchor_rms:.2f} px',
    f'Final spine RMS:   {res_2step["spine_rms"]:.2f} px',
    f'Final L1 error:    {res_2step["L1_err"]:.2f} px',
    f'Final L6 error:    {res_2step["L6_err"]:.2f} px',
    f'Final P1 error:    {res_2step["P1_err"]:.2f} px' if res_2step['P1_err'] is not None else 'Final P1 error:    --',
    f'Final L 2-pt RMS:  {res_2step["L_2pt_rms"]:.2f} px',
    '',
    'Reading order:',
    '1. overlay both original grids',
    '2. fit affine on 8 anchors',
    '3. inspect affine grid result',
    '4. use TPS control points after Step 1',
    '5. inspect the final two-step grid overlay',
]
ax.text(0.02, 0.98, '\n'.join(summary_lines), va='top', ha='left', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.95))

plt.tight_layout()
method4_path = OUTPUT_DIR / 'method4_step_by_step.png'
fig.savefig(method4_path, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved: {method4_path}')
